# Week 38, Day 4: Motion: space-time filters and the dorsal stream

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w038/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Direction from space-time orientation

Write oriented(fx, ft), the space-time Gabor exp(-x²/2σx² - t²/2σt²) cos(2π(fx x - ft t)) on the grid (X in degrees, T in seconds). The separable filter, the stimuli and the drifting-grating response (mean over phases of the rectified dot product) are given. Then compute the direction index (R_pref - R_null)/(R_pref + R_null) for both filters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
x = np.linspace(-2, 2, 80); t = np.linspace(-0.15, 0.15, 60)
X, T = np.meshgrid(x, t)                    # rows are time, columns are space
sx, st = 0.5, 0.05

def separable(fx, ft):
    return np.exp(-X ** 2 / (2 * sx ** 2) - T ** 2 / (2 * st ** 2)) * np.cos(2 * np.pi * fx * X) * np.cos(2 * np.pi * ft * T)

def oriented(fx, ft):
    return ...

def drifting(fx, ft, direction, phase):
    return np.cos(2 * np.pi * (fx * X - direction * ft * T) + phase)

def drift_resp(rf, fx, ft, direction):
    return np.mean([max(0.0, np.sum(rf * drifting(fx, ft, direction, p))) for p in np.linspace(0, 2 * np.pi, 8, endpoint=False)])

dsi = lambda r, l: abs(r - l) / (r + l)
fx, ft = 1.0, 8.0                           # 1 cycle/deg, 8 Hz: preferred speed 8 deg/s
sep_r, sep_l = drift_resp(separable(fx, ft), fx, ft, 1), drift_resp(separable(fx, ft), fx, ft, -1)
ori_r, ori_l = drift_resp(oriented(fx, ft), fx, ft, 1), drift_resp(oriented(fx, ft), fx, ft, -1)
print(f"separable: right {sep_r:.1f}, left {sep_l:.1f}, DSI {dsi(sep_r, sep_l):.2f}")
print(f"oriented:  right {ori_r:.1f}, left {ori_l:.1f}, DSI {dsi(ori_r, ori_l):.2f}")
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
ax[0].imshow(separable(fx, ft), cmap="RdBu_r", aspect="auto", extent=[-2, 2, 0.15, -0.15]); ax[0].set_title("separable")
ax[1].imshow(oriented(fx, ft), cmap="RdBu_r", aspect="auto", extent=[-2, 2, 0.15, -0.15]); ax[1].set_title("oriented in space-time")
for a in ax: a.set_xlabel("space (deg)"); a.set_ylabel("time (s)")
plt.show()

In [ ]:
import numpy as np
def test_separable():
    "the separable filter responds equally to both directions (direction index below 0.05)"
    assert dsi(sep_r, sep_l) < 0.05

def test_oriented():
    "the oriented filter prefers rightward motion with a direction index above 0.8"
    assert dsi(ori_r, ori_l) > 0.8

test_separable()
test_oriented()
print("All checks passed")

### Exercise: Opponent motion energy

Write motion_energy(stim, fx, ft, direction): the sum of squared dot products of the stimulus with an even and an odd space-time filter oriented for that direction (phase 0 and π/2 in cos(2π(fx x - direction ft t)) + phase). Then compute rightward energy across phases, and opponent energy (right minus left) for rightward and leftward gratings.

In [ ]:
def oriented_dir(fx, ft, direction, phase):
    return np.exp(-X ** 2 / (2 * sx ** 2) - T ** 2 / (2 * st ** 2)) * np.cos(2 * np.pi * (fx * X - direction * ft * T) + phase)

def motion_energy(stim, fx, ft, direction):
    ...

phases = np.linspace(0, 2 * np.pi, 16, endpoint=False)
e_right = np.array([motion_energy(drifting(fx, ft, 1, p), fx, ft, 1) for p in phases])
opp = lambda stim: motion_energy(stim, fx, ft, 1) - motion_energy(stim, fx, ft, -1)
opp_right, opp_left = opp(drifting(fx, ft, 1, 0.3)), opp(drifting(fx, ft, -1, 0.3))
print(f"rightward energy across phases: mean {e_right.mean():.0f}, relative spread {e_right.std() / e_right.mean():.4f}")
print(f"opponent energy: rightward grating {opp_right:.0f}, leftward grating {opp_left:.0f}")

In [ ]:
import numpy as np
def test_steady():
    "rightward motion energy varies by less than 1% across grating phases"
    assert e_right.std() / e_right.mean() < 0.01

def test_opponent():
    "opponent energy is positive for rightward motion and negative for leftward"
    assert opp_right > 0 and opp_left < 0 and abs(opp_right + opp_left) < 0.01 * abs(opp_right)

test_steady()
test_opponent()
print("All checks passed")

### Exercise: Speed tuning

For speeds from 1 to 24 deg/s, compute the rightward motion energy of the (fx = 1, ft = 8) filters for a rightward grating of spatial frequency 1 cycle/deg (temporal frequency = speed times spatial frequency). Repeat with gratings of 2 cycles/deg.

In [ ]:
speeds = np.arange(1, 25)
speed_tuning = ...
speed_tuning_2 = ...
print(f"preferred speed: {speeds[np.argmax(speed_tuning)]} deg/s with 1 c/deg gratings, {speeds[np.argmax(speed_tuning_2)]} deg/s with 2 c/deg gratings")
plt.plot(speeds, speed_tuning / speed_tuning.max(), label="1 c/deg"); plt.plot(speeds, speed_tuning_2 / speed_tuning_2.max(), label="2 c/deg")
plt.xlabel("grating speed (deg/s)"); plt.ylabel("normalised motion energy"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_peak():
    "motion energy peaks within 2 deg/s of the filter's preferred speed of 8 deg/s"
    assert abs(speeds[np.argmax(speed_tuning)] - 8) <= 2

def test_tf():
    "with gratings of 2 cycles/deg, the preferred speed roughly halves"
    assert speeds[np.argmax(speed_tuning_2)] < 0.65 * speeds[np.argmax(speed_tuning)]

test_peak()
test_tf()
print("All checks passed")